# Chapter 3: Measuring Similarity — Kernels on the Grid

**What you'll learn:** How to compute layer and temporal kernels, visualise the three computational phases, and detect phase transitions.

**Key concept:** Kernels measure similarity between layers (or tokens). The layer kernel reveals three phases: setup, processing, decision.

**Time:** ~15 minutes

## 3.1 Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="auto")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 3.2 The Layer Kernel

The **layer kernel** measures how similar two layers are. For each pair of layers (l, l'), we compute the average dot product of their token representations:

$$K_{\text{layer}}(l, l') = \frac{1}{T} \text{tr}(\tilde{H}^{(l)\top} \tilde{H}^{(l')})$$

High values = similar representations. Low values = the model has significantly transformed the representation between those layers.

In [2]:
result = ltg.analyse("Explain how photosynthesis converts sunlight into chemical energy", model=model, compute_dependency=False)

# The layer kernel is already computed
K = result.layer_kernel_matrix
print(f"Layer kernel shape: {K.shape} = ({result.n_layers} × {result.n_layers})")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(K, cmap='viridis', origin='lower')
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Layer', fontsize=12)
ax.set_title('Layer Kernel — Similarity Between Layers', fontsize=14)
plt.colorbar(im, ax=ax, label='Similarity')
plt.tight_layout()
plt.show()

Layer kernel shape: (28, 28) = (28 × 28)


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60184/1939933409.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.3 The Three Computational Phases

Look at the heatmap above. You should see a **block-diagonal structure**:
- **Top-left block (early layers):** High mutual similarity → setup phase
- **Transition zone (middle):** Similarity drops → processing phase
- **Bottom-right block (late layers):** Similarity rises again → decision phase

Let's detect the phase boundaries automatically.

In [3]:
# Consecutive layer similarity: how similar is layer l to layer l+1?
consecutive_sim = np.array([K[l, l+1] for l in range(K.shape[0] - 1)])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(consecutive_sim, 'o-', color='#4477AA', linewidth=2, markersize=4)
ax.set_xlabel('Layer l', fontsize=12)
ax.set_ylabel('K(l, l+1)', fontsize=12)
ax.set_title('Consecutive Layer Similarity — Phase Transitions', fontsize=14)

# Find phase boundaries (local minima)
from scipy.signal import argrelmin
minima = argrelmin(consecutive_sim, order=3)[0]
for m in minima:
    ax.axvline(m, color='red', linestyle='--', alpha=0.5)
    ax.annotate(f'Transition\nlayer {m}', xy=(m, consecutive_sim[m]),
                xytext=(m+1, consecutive_sim.max()*0.9),
                fontsize=9, color='red',
                arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print(f"Phase transitions at layers: {list(minima)}")
print(f"Total layers: {result.n_layers}")
print(f"Transition at: {[f'{m/result.n_layers:.0%}' for m in minima]} depth")

Phase transitions at layers: [np.int64(7), np.int64(12)]
Total layers: 28
Transition at: ['25%', '43%'] depth


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60184/1634511259.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Temporal Kernel — Which Tokens Are Similar?

We can also compute similarity between *tokens* instead of layers. This shows which tokens develop similar representations.

In [4]:
# Compute temporal kernel at the final layer
H_w = result.H_whitened
L, T, k = H_w.shape

K_time = H_w[-1] @ H_w[-1].T  # (T, T) at last layer
K_time_norm = K_time / (np.linalg.norm(H_w[-1], axis=1, keepdims=True) @ 
                         np.linalg.norm(H_w[-1], axis=1, keepdims=True).T + 1e-8)

tokens = [model.tokenizer.decode([tid]) for tid in model.tokenizer.encode(
    "Explain how photosynthesis converts sunlight into chemical energy")]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(K_time_norm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens, fontsize=8)
ax.set_title('Temporal Kernel at Final Layer (cosine similarity)', fontsize=14)
plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.show()

/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60184/242389710.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Comparing Kernels Across Prompts

Does the three-phase structure persist across different types of prompts?

In [5]:
prompts = {
    'Arithmetic': 'What is 17 times 23?',
    'Reasoning': 'If all birds can fly and penguins are birds, can penguins fly?',
    'Retrieval': 'What is the capital of Japan?',
    'Creative': 'Write a haiku about mountains',
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, text) in zip(axes, prompts.items()):
    r = ltg.analyse(text, model=model, compute_dependency=False)
    K = r.layer_kernel_matrix
    ax.imshow(K, cmap='viridis', origin='lower')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Layer')
    if ax == axes[0]:
        ax.set_ylabel('Layer')

plt.suptitle('Layer Kernels Across Task Types', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("Notice: the block-diagonal structure appears in ALL task types.")

Notice: the block-diagonal structure appears in ALL task types.


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60184/3063739773.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Key Takeaways

1. **The layer kernel** reveals a universal three-phase computational structure.
2. **Phase transitions** can be detected automatically from drops in consecutive similarity.
3. **The temporal kernel** shows which tokens develop similar representations.
4. **The three-phase structure is task-independent** — it appears for arithmetic, reasoning, retrieval, and creative tasks.

## Exercise

Compute the layer kernel for a very short prompt (3 tokens) and a very long prompt (50+ tokens). Does the phase transition occur at the same *relative* depth (e.g., 25% and 75%)?

In [6]:
# Your turn!
# r_short = ltg.analyse("Hello world", model=model, compute_dependency=False)
# r_long = ltg.analyse("A long prompt...", model=model, compute_dependency=False)
# Compare K matrices and find transitions